In [ ]:
# =================================================================
# S0 -- CONFIGURATION DE L'ENTRAÎNEMENT
# =================================================================
import os
import sys
from pathlib import Path

MODE = "train"  # Mode entraînement actif
DEBUG = True   # Passer à True pour tester le pipeline sur un mini-batch rapide

# Configuration de l'Ensemble de pseudo-labeling
CONF_THRESHOLD = 0.85  # Seuil de confiance strict pour le filtrage
ACTIVE_SOURCES = ["focal", "sc", "pseudo_sc"]
SHARES = {"focal": 0.7, "sc": 0.1, "pseudo_sc": 0.2}  # Batch composition
SOURCE_WEIGHTS = {
    "focal":         1.0,
    "focal_missing": 0.0,
    "sc":            1.0,
    "pseudo_sc":     1.0,
}

# =================================================================
# S1 -- INSTALLATION SILENCIEUSE DES DEPENDANCES
# =================================================================
if MODE == "train":
    print("Installation des bibliothèques...")
    !pip install -q timm torchaudio onnxscript onnx onnxruntime --quiet --no-warn-conflicts
    wheel_path = "/kaggle/input/datasets/hellodave2035/birdclef-perch-models/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl"
    if os.path.exists(wheel_path):
        !pip install -q --no-deps {wheel_path} --quiet --no-warn-conflicts

# =================================================================
# S2 -- IMPORTS & INITIALISATION (CORRIGÉ)
# =================================================================
import time
import json
import pickle
import gc
import random
import math
import re
import glob
import numpy as np
import pandas as pd
from collections import defaultdict
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.cuda.amp import GradScaler, autocast
import torchaudio
import timm
from sklearn.metrics import roc_auc_score
# --- CORRECTIF : Importation de StratifiedKFold ET GroupKFold ---
from sklearn.model_selection import StratifiedKFold, GroupKFold
import onnxruntime as ort
import soundfile as sf
import librosa
import warnings
warnings.filterwarnings("ignore")

SEED = 42
def seed_everything(s=42):
    random.seed(s)
    os.environ["PYTHONHASHSEED"] = str(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device actif : {device}")

# Chemins d'accès
COMP_DIR = Path("/kaggle/input/competitions/birdclef-2026")
WAVEFORM_CACHE_DIR = Path("/kaggle/input/datasets/tuckerarrants/birdclef-2026-waveform-cache/waveform_cache")
PERCH_ONNX_PATH = Path("/kaggle/input/datasets/hellodave2035/birdclef-perch-models/perch_v2.onnx")
OUT_DIR = Path("/kaggle/working")

NUM_CLASSES = 234
SR = 32000
TRAIN_DURATION = 5
TRAIN_SAMPLES  = SR * TRAIN_DURATION
VAL_DURATION   = 5
VAL_SAMPLES    = SR * VAL_DURATION
N_FOLDS = 5

N_FFT      = 2048
HOP_LENGTH = 512
N_MELS     = 256
FMIN       = 20
FMAX       = 16000
BACKBONE_NAME = "tf_efficientnet_b0.ns_jft_in1k"  
USE_PERCH_DISTILL = True
PERCH_EMBED_DIM   = 1536
ALPHA_DISTILL     = 1.0

FOLDS  = [0, 1, 2, 3, 4]
EPOCHS = 20  
BATCH  = 16 if DEBUG else 64
LR     = 3e-4  
MIN_LR = 1e-6
WD     = 1e-4
WARMUP_EPOCHS = 1
MIN_SAMPLE = 20
AUG_PROB = 0.5
AUG_GAIN_DB_RANGE      = (-6.0, 6.0)
AUG_NOISE_SNR_DB_RANGE = (10.0, 30.0)

USE_FOCAL_MIXUP = True
MIXUP_PROB = 0.5
MIXUP_ALPHA = 0.4
MIXUP_HARD = True
USE_FOCAL_SC_MIXUP = True
FOCAL_SC_MIXUP_PROB = 0.5
FOCAL_SC_MIXUP_ALPHA = 0.4

FREQ_MASK_PARAM = 10
TIME_MASK_PARAM = 10
NUM_FREQ_MASKS  = 1
NUM_TIME_MASKS  = 2

# Chargement métadonnées de base
sample_sub = pd.read_csv(COMP_DIR / "sample_submission.csv")
PRIMARY_LABELS = sample_sub.columns[1:].tolist()
LABEL2IDX = {label: idx for idx, label in enumerate(PRIMARY_LABELS)}
taxonomy = pd.read_csv(COMP_DIR / "taxonomy.csv")
label_to_taxon = dict(zip(taxonomy["primary_label"].astype(str), taxonomy["class_name"].astype(str)))
TAXON_MASKS = {t: np.array([i for i, l in enumerate(PRIMARY_LABELS) if label_to_taxon.get(l, "") == t])
               for t in ["Aves", "Amphibia", "Insecta", "Mammalia", "Reptilia"]}

audio_cache_meta = pd.read_csv(WAVEFORM_CACHE_DIR / "audio_cache_meta.csv")
train_df = pd.read_csv(COMP_DIR / "train.csv")
audio_cache_meta = audio_cache_meta.merge(train_df[["filename", "secondary_labels"]], on="filename", how="left")
audio_cache_meta = audio_cache_meta[audio_cache_meta["primary_label"].isin(LABEL2IDX)].reset_index(drop=True)

sc_cache_meta = pd.read_csv(WAVEFORM_CACHE_DIR / "soundscape_cache_meta.csv")
sc_cache_meta["label_list"] = sc_cache_meta["label_list"].apply(lambda x: x.split(";") if isinstance(x, str) else [])

sc_labels_raw = pd.read_csv(COMP_DIR / "train_soundscapes_labels.csv").drop_duplicates()
sc_labels_raw["start_sec"] = pd.to_timedelta(sc_labels_raw["start"]).dt.total_seconds().astype(int)

Y_SC = np.zeros((len(sc_cache_meta), NUM_CLASSES), dtype=np.float32)
for i, row in sc_cache_meta.iterrows():
    matches = sc_labels_raw[(sc_labels_raw["filename"] == row["filename"]) & (sc_labels_raw["start_sec"] == row["start_sec"])]
    for _, m in matches.iterrows():
        for lbl in str(m["primary_label"]).split(";"):
            lbl = lbl.strip()
            if lbl in LABEL2IDX:
                Y_SC[i, LABEL2IDX[lbl]] = 1.0

labeled_sc_mask = Y_SC.sum(axis=1) > 0

# Splits Folds
sc_files = sc_cache_meta[["filename", "site"]].drop_duplicates().reset_index(drop=True)
gkf = GroupKFold(n_splits=N_FOLDS)
sc_files["fold"] = -1
for fold, (_, val_idx) in enumerate(gkf.split(sc_files, groups=sc_files["filename"])):
    sc_files.loc[sc_files.index[val_idx], "fold"] = fold
file_to_fold = dict(zip(sc_files["filename"], sc_files["fold"]))
sc_cache_meta["fold"] = sc_cache_meta["filename"].map(file_to_fold).fillna(-1).astype(int)

audio_for_split = audio_cache_meta.drop_duplicates("original_idx").reset_index(drop=True)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
audio_for_split["fold"] = -1
for fold, (_, val_idx) in enumerate(skf.split(audio_for_split, audio_for_split["primary_label"])):
    audio_for_split.loc[val_idx, "fold"] = fold
audio_cache_meta = audio_cache_meta.merge(audio_for_split[["original_idx", "fold"]], on="original_idx", how="left")

# =================================================================
# S3 -- GENERATION DE PSEUDO-LABELS & SAUVEGARDE EN WORKSPACE
# =================================================================
def find_sed_dir():
    hits = sorted(Path("/kaggle/input").rglob("sed_fold0.onnx"))
    if not hits:
        raise FileNotFoundError("sed_fold0.onnx introuvable. Veuillez lier tuckerarrants/bc2026-distilled-sed-public.")
    return hits[0].parent

def make_sed_session(path):
    so = ort.SessionOptions()
    so.intra_op_num_threads = 4
    return ort.InferenceSession(str(path), sess_options=so, providers=["CUDAExecutionProvider", "CPUExecutionProvider"])

def audio_to_mel(chunks):
    mels = []
    for x in chunks:
        s = librosa.feature.melspectrogram(y=x, sr=SR, n_fft=2048, hop_length=512, n_mels=256, fmin=20, fmax=16000, power=2.0)
        s = librosa.power_to_db(s, top_db=80)
        s = (s - s.mean()) / (s.std() + 1e-6)
        mels.append(s)
    return np.stack(mels)[:, None].astype(np.float32)

def sigmoid_sed(x):
    return (1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))).astype(np.float32)

# Répertoire de cache local pour les pseudo-labels
PSEUDO_CACHE_DIR = OUT_DIR / "pseudo_cache"
PSEUDO_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# 1. Scanner tous les fichiers soundscapes physiques
all_ogg_files = sorted(glob.glob(str(COMP_DIR / "train_soundscapes" / "*.ogg")))
labeled_filenames = set(sc_labels_raw["filename"].unique())
unlabeled_ogg_files = [f for f in all_ogg_files if os.path.basename(f) not in labeled_filenames]

if DEBUG:
    unlabeled_ogg_files = unlabeled_ogg_files[:2]

print(f"Trouvé {len(unlabeled_ogg_files)} soundscapes entièrement non-étiquetés pour le pseudo-labeling.")

# 2. Inférence de l'ensemble SED ONNX
sed_dir = find_sed_dir()
sed_fold_paths = sorted(sed_dir.glob("sed_fold*.onnx"))
sed_sessions = [make_sed_session(p) for p in sed_fold_paths]

pseudo_metadata_rows = []
chunk_counter = 0

for file_path in tqdm(unlabeled_ogg_files, desc="Pseudo-labeling soundscapes"):
    filename = os.path.basename(file_path)
    try:
        y, sr0 = sf.read(str(file_path), dtype="float32", always_2d=False)
        if y.ndim > 1:
            y = y.mean(axis=1)
        if sr0 != SR:
            y = librosa.resample(y, orig_sr=sr0, target_sr=SR)
    except Exception as e:
        print(f"Erreur de chargement {filename}: {e}")
        continue
        
    target_len = 60 * SR
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]
        
    chunks = y.reshape(12, TRAIN_SAMPLES)
    mel = audio_to_mel(chunks)
    
    p_sum = np.zeros((12, NUM_CLASSES), dtype=np.float32)
    for sess in sed_sessions:
        outs = sess.run(None, {sess.get_inputs()[0].name: mel})
        clip_logits = outs[0]
        frame_max   = outs[1].max(axis=1)
        p_sum += 0.5 * sigmoid_sed(clip_logits) + 0.5 * sigmoid_sed(frame_max)
    p_mean = p_sum / len(sed_sessions)
    
    file_fold = int(hash(filename) % N_FOLDS)
    
    # Validation du seuil de confiance et mise en cache
    for win_idx in range(12):
        probs = p_mean[win_idx]
        if probs.max() > CONF_THRESHOLD:
            active_labels = (probs > CONF_THRESHOLD).astype(np.float32)
            
            # Enregistrement au format int16 compressé identique au cache d'origine
            chunk_data = chunks[win_idx]
            tensor_int16 = torch.from_numpy((chunk_data * 32767.0).astype(np.int16))
            
            cache_filename = f"pseudo_chunk_{chunk_counter}.pt"
            torch.save(tensor_int16, PSEUDO_CACHE_DIR / cache_filename)
            
            pseudo_metadata_rows.append({
                "cache_file": f"pseudo_cache/{cache_filename}",
                "start_sec": 0,
                "label_vector": active_labels,
                "fold": file_fold,
                "filename": filename
            })
            chunk_counter += 1

if len(pseudo_metadata_rows) > 0:
    pseudo_sc_df = pd.DataFrame(pseudo_metadata_rows)
    Y_PSEUDO = np.stack(pseudo_sc_df["label_vector"].values)
    print(f"Pseudo-labeled dataset généré avec succès : {Y_PSEUDO.shape}")
else:
    pseudo_sc_df = None
    Y_PSEUDO = None
    print("Aucun pseudo-label généré sous le seuil de confiance spécifié.")

# =================================================================
# S4 -- ARCHITECTURE ET DATASETS
# =================================================================
class MelSpecTransform(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
            n_mels=N_MELS, f_min=FMIN, f_max=FMAX, power=2.0,
        )
        self.db_transform = torchaudio.transforms.AmplitudeToDB(top_db=80)
    def forward(self, waveform):
        return self.db_transform(self.mel_spec(waveform))

class SpecAugment(nn.Module):
    def __init__(self):
        super().__init__()
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=FREQ_MASK_PARAM)
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=TIME_MASK_PARAM)
    def forward(self, mel):
        for _ in range(NUM_FREQ_MASKS):
            mel = self.freq_mask(mel)
        for _ in range(NUM_TIME_MASKS):
            mel = self.time_mask(mel)
        return mel

class PerchTeacher:
    def __init__(self, onnx_path, device_str="cuda"):
        providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if device_str == "cuda" else ["CPUExecutionProvider"]
        self.session = ort.InferenceSession(str(onnx_path), providers=providers)
        self.input_name = self.session.get_inputs()[0].name
        
        # --- CORRECTIF : Sélectionner le tenseur global 2D ['batch', 1536] ---
        self._embed_idx = None
        for i, o in enumerate(self.session.get_outputs()):
            if o.shape and len(o.shape) == 2 and o.shape[-1] == PERCH_EMBED_DIM:
                self._embed_idx = i
                break
        if self._embed_idx is None:
            self._embed_idx = 1
        print(f"Perch ONNX chargé (Embed Index = {self._embed_idx})")
        
    @torch.no_grad()
    def embed(self, waveforms_5s):
        wav_np = waveforms_5s.cpu().numpy()
        results = self.session.run(None, {self.input_name: wav_np})
        return torch.from_numpy(results[self._embed_idx]).float()

class DistillHead(nn.Module):
    def __init__(self, backbone_dim, embed_dim=1536):
        super().__init__()
        self.proj = nn.Linear(backbone_dim, embed_dim)
    def forward(self, feature_map):
        gap = feature_map.mean(dim=[2, 3])
        return self.proj(gap)

class GeMFreqPool(nn.Module):
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(float(p_init)))
        self.eps = eps
    def forward(self, x):
        p = self.p.clamp(min=1.0)
        x = x.clamp(min=self.eps).pow(p)
        x = x.mean(dim=2)
        return x.pow(1.0 / p)

class BirdSEDModel(nn.Module):
    def __init__(self, backbone_name=BACKBONE_NAME, num_classes=NUM_CLASSES, drop_path_rate=0.1, hidden_dim=512):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=True, in_chans=1, num_classes=0, global_pool="", drop_path_rate=drop_path_rate)
        self.backbone_dim = 1280
        self.gem_freq = GeMFreqPool(p_init=3.0)
        self.dense = nn.Sequential(nn.Dropout(0.25), nn.Linear(self.backbone_dim, hidden_dim), nn.ReLU(inplace=True), nn.Dropout(0.5))
        self.att = nn.Conv1d(hidden_dim, num_classes, kernel_size=1, bias=True)
        self.cla = nn.Conv1d(hidden_dim, num_classes, kernel_size=1, bias=True)
        if USE_PERCH_DISTILL:
            self.distill_head = DistillHead(self.backbone_dim, PERCH_EMBED_DIM)

    def forward(self, x, return_framewise=False, return_distill=False):
        h = self.backbone(x)
        distill_emb = None
        if return_distill and hasattr(self, 'distill_head'):
            distill_emb = self.distill_head(h)
        h_cls = h.detach() if USE_PERCH_DISTILL else h
        h_cls = self.gem_freq(h_cls)
        h_cls = h_cls.permute(0, 2, 1)
        h_cls = self.dense(h_cls)
        h_cls = h_cls.permute(0, 2, 1)
        norm_att = torch.softmax(torch.tanh(self.att(h_cls)), dim=-1)
        framewise_logits = self.cla(h_cls)
        clip_logits = torch.sum(norm_att * framewise_logits, dim=2)
        fw = framewise_logits.permute(0, 2, 1) if return_framewise else None
        if return_framewise and return_distill:
            return clip_logits, fw, distill_emb
        elif return_framewise:
            return clip_logits, fw
        elif return_distill:
            return clip_logits, distill_emb
        return clip_logits

def load_int16(path):
    """Charge un tenseur int16 sauvegardé avec torch.save."""
    return torch.load(str(path), weights_only=True)

def load_sc_waveform_from(cache_dir, cache_file):
    pp = cache_dir / cache_file
    if not pp.exists():
        return None
    return load_int16(pp).numpy()

def extract_chunk_np(waveform, start_sample, n_samples):
    total = len(waveform)
    if total <= n_samples:
        return np.pad(waveform, (n_samples - total, 0))
    end = start_sample + n_samples
    if end > total:
        start_sample = max(0, total - n_samples)
    return waveform[start_sample:start_sample + n_samples]

def apply_aug(w):
    if np.random.random() < AUG_PROB:
        w = w * (10 ** (np.random.uniform(*AUG_GAIN_DB_RANGE) / 20))
    if np.random.random() < AUG_PROB:
        sp = (w ** 2).mean()
        if sp > 1e-10:
            w = w + np.random.randn(*w.shape).astype(w.dtype) * np.sqrt(sp / (10 ** (np.random.uniform(*AUG_NOISE_SNR_DB_RANGE) / 10)))
    return w

class FocalDS(Dataset):
    def __init__(self, df, l2i, secondary_lookup=None, sc_mixup_sources=None, fold_k=None, aug=False):
        self.df, self.l2i, self.aug = df.reset_index(drop=True), l2i, aug
        self.secondary_lookup = secondary_lookup
        self.sc_mixup_sources = sc_mixup_sources
        self.fold_k = fold_k
    def __len__(self):
        return len(self.df)
    def _load_chunk(self, r):
        w = load_sc_waveform_from(WAVEFORM_CACHE_DIR, r["cache_file"])
        if w is None:
            return None, None
        if self.aug and len(w) > TRAIN_SAMPLES:
            start = np.random.randint(0, len(w) - TRAIN_SAMPLES + 1)
        else:
            start = 0
        ch = extract_chunk_np(w, start, TRAIN_SAMPLES)
        lb = np.zeros(NUM_CLASSES, dtype=np.float32)
        if str(r["primary_label"]) in self.l2i:
            lb[self.l2i[str(r["primary_label"])]] = 1.0
        return ch, lb
    def __getitem__(self, i):
        r = self.df.iloc[i]
        ch1, lb1 = self._load_chunk(r)
        if ch1 is None:
            return (torch.zeros(1, TRAIN_SAMPLES), torch.zeros(NUM_CLASSES), torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "focal_missing")
        if self.aug:
            ch1 = apply_aug(ch1)
        return (torch.from_numpy(ch1.astype(np.float32)).unsqueeze(0), torch.from_numpy(lb1), torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "focal")

class ScDS(Dataset):
    def __init__(self, Y, sc_df, aug=False):
        self.Y, self.df, self.aug = Y, sc_df.reset_index(drop=True), aug
    def __len__(self):
        return len(self.Y)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        wav_full = load_sc_waveform_from(WAVEFORM_CACHE_DIR, row["cache_file"])
        if wav_full is None:
            chunk = np.zeros(TRAIN_SAMPLES, dtype=np.float32)
        else:
            chunk = extract_chunk_np(wav_full, int(row["start_sec"]) * SR, TRAIN_SAMPLES)
        if self.aug:
            chunk = apply_aug(chunk)
        return (torch.from_numpy(chunk.astype(np.float32)).unsqueeze(0), torch.from_numpy(self.Y[i].astype(np.float32)), torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "sc")

# Nouveau Dataset pour charger les signaux pseudo-étiquetés pré-découpés depuis /kaggle/working
class PseudoScDS(Dataset):
    def __init__(self, Y, df, aug=False):
        self.Y, self.df, self.aug = Y, df.reset_index(drop=True), aug
    def __len__(self):
        return len(self.Y)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        wav_full = load_sc_waveform_from(OUT_DIR, row["cache_file"])
        if wav_full is None:
            chunk = np.zeros(TRAIN_SAMPLES, dtype=np.float32)
        else:
            chunk = extract_chunk_np(wav_full, int(row["start_sec"]) * SR, TRAIN_SAMPLES)
        if self.aug:
            chunk = apply_aug(chunk)
        return (torch.from_numpy(chunk.astype(np.float32)).unsqueeze(0), torch.from_numpy(self.Y[i].astype(np.float32)), torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "pseudo_sc")

class MixSamp(torch.utils.data.Sampler):
    def __init__(self, sizes, names, shares, bs, nst, seed=0):
        self.sizes, self.names, self.bs, self.nst = sizes, names, bs, nst
        self.rng = np.random.default_rng(seed)
        per_src = [max(1, int(round(bs * shares.get(n, 0.0)))) for n in names]
        total = sum(per_src)
        if total != bs:
            per_src[int(np.argmax(per_src))] += (bs - total)
        self.per_src = per_src
        self.offsets = [0]
        for s in sizes[:-1]:
            self.offsets.append(self.offsets[-1] + s)
    def __len__(self):
        return self.nst
    def __iter__(self):
        for _ in range(self.nst):
            batch = []
            for off, size, n in zip(self.offsets, self.sizes, self.per_src):
                if n <= 0 or size <= 0:
                    continue
                idxs = self.rng.integers(0, size, size=n)
                batch.extend([off + int(i) for i in idxs])
            self.rng.shuffle(batch)
            yield batch

def collate_m(batch):
    return (torch.stack([b[0] for b in batch]), torch.stack([b[1] for b in batch]), torch.stack([b[2] for b in batch]), torch.stack([b[3] for b in batch]), [b[4] for b in batch])

def mk_sw(sr):
    return torch.tensor([SOURCE_WEIGHTS.get(s, 0.0) for s in sr], dtype=torch.float32)

# =================================================================
# S5 -- BOUCLE D'ENTRAÎNEMENT & ADAPTATION DE DOMAINE
# =================================================================
def build_active_datasets(fold_k):
    items = []
    # 1. Focal
    fds = FocalDS(audio_cache_meta[audio_cache_meta["fold"] != fold_k], LABEL2IDX, aug=True)
    items.append(("focal", fds, len(fds)))
    
    # 2. Soundscapes experts
    vm = sc_cache_meta["fold"].values == fold_k
    sds = ScDS(Y_SC[~vm], sc_cache_meta[~vm], aug=True)
    items.append(("sc", sds, len(sds)))
    
    # 3. Soundscapes Pseudo-étiquetés
    if pseudo_sc_df is not None:
        train_pseudo_mask = (pseudo_sc_df["fold"] != fold_k)
        if train_pseudo_mask.sum() > 0:
            pseudo_df = pseudo_sc_df[train_pseudo_mask].reset_index(drop=True)
            Y_pseudo_train = Y_PSEUDO[train_pseudo_mask]
            psds = PseudoScDS(Y_pseudo_train, pseudo_df, aug=True)
            items.append(("pseudo_sc", psds, len(psds)))
            
    return items

def train_domain_adaptation(fold_k):
    print(f"\nEntraînement Fold {fold_k}...")
    active = build_active_datasets(fold_k)
    names, datasets, sizes = zip(*active)
    mds = ConcatDataset(list(datasets))
    nst = max(100, int(sum(sizes) / BATCH))
    
    m = BirdSEDModel(BACKBONE_NAME).to(device)
    mel_transform = MelSpecTransform().to(device)
    spec_augment = SpecAugment().to(device)
    perch_teacher = PerchTeacher(PERCH_ONNX_PATH, "cuda") if USE_PERCH_DISTILL else None
    
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-4)
    scaler = GradScaler()
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=nst * EPOCHS, eta_min=MIN_LR)
    
    best_loss = float("inf")
    best_state = None
    
    for ep in range(EPOCHS):
        m.train()
        smp = MixSamp(list(sizes), list(names), SHARES, BATCH, nst, seed=42 + ep)
        tl = DataLoader(mds, batch_sampler=smp, collate_fn=collate_m, num_workers=0, pin_memory=True)
        el, el_cls, el_dist, nb_count = 0.0, 0.0, 0.0, 0
        
        for wav, lb, wt, mk, sr in tl:
            wav, lb, wt, mk = wav.to(device), lb.to(device), wt.to(device), mk.to(device)
            sw = mk_sw(sr).to(device)
            
            with torch.no_grad():
                mel = mel_transform(wav)
                for idx in range(mel.size(0)):
                    mel[idx] = (mel[idx] - mel[idx].mean()) / (mel[idx].std() + 1e-6)
                mel = spec_augment(mel)
                
            with autocast():
                clip_logits, framewise, distill_emb = m(mel, return_framewise=True, return_distill=True)
                frame_max_logits = framewise.max(dim=1).values
                
                bce_clip = F.binary_cross_entropy_with_logits(clip_logits, lb, reduction="none")
                bce_frame = F.binary_cross_entropy_with_logits(frame_max_logits, lb, reduction="none")
                bce = 0.5 * bce_clip + 0.5 * bce_frame
                ps = (bce * wt * mk).sum(1) / (mk.sum(1) + 1e-8)
                cls_loss = (ps * sw).mean()
                
                with torch.no_grad():
                    wav_5s = wav.squeeze(1)
                perch_emb = perch_teacher.embed(wav_5s).to(device)
                
                # --- CORRECTIF : Alignement dimensionnel des embeddings ---
                distill_loss = F.mse_loss(distill_emb, perch_emb)
                loss = cls_loss + ALPHA_DISTILL * distill_loss
                
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sch.step()
            
            # --- CORRECTIF : Dissociation des instructions sur plusieurs lignes ---
            el += loss.item()
            el_cls += cls_loss.item()
            el_dist += distill_loss.item()
            nb_count += 1
            
        print(f"  Ep{ep:02d}: loss={el/nb_count:.4f} cls={el_cls/nb_count:.4f} dist={el_dist/nb_count:.4f}")
        
        if el/nb_count < best_loss:
            best_loss = el/nb_count
            best_state = {k: v.cpu().clone() for k, v in m.state_dict().items()}
            
    # Exportation ONNX
    print(f"  Exportation du modèle ONNX optimisé pour le Fold {fold_k}...")
    torch.save(best_state, OUT_DIR / f"fold{fold_k}_best_adaptation.pt")
    
    m.load_state_dict(best_state)
    m.eval()
    
    class SEDExportWrapper(nn.Module):
        def __init__(self, backbone_name, num_classes, backbone_dim, hidden_dim=512):
            super().__init__()
            self.backbone = timm.create_model(backbone_name, pretrained=False, in_chans=1, num_classes=0, global_pool="", drop_path_rate=0.1)
            self.gem_freq = GeMFreqPool(p_init=3.0)
            self.dense_drop1 = nn.Dropout(0.25)
            self.dense_conv = nn.Conv1d(backbone_dim, hidden_dim, 1)
            self.dense_relu = nn.ReLU(inplace=True)
            self.dense_drop2 = nn.Dropout(0.5)
            self.att = nn.Conv1d(hidden_dim, num_classes, 1)
            self.cla = nn.Conv1d(hidden_dim, num_classes, 1)
        def forward(self, mel):
            h = self.backbone(mel)
            h = self.gem_freq(h)
            h = self.dense_drop1(h)
            h = self.dense_conv(h)
            h = self.dense_relu(h)
            h = self.dense_drop2(h)
            norm_att = torch.softmax(torch.tanh(self.att(h)), dim=-1)
            framewise = self.cla(h)
            clip = torch.sum(norm_att * framewise, dim=2)
            return clip, framewise.permute(0, 2, 1)

    export_model = SEDExportWrapper(BACKBONE_NAME, NUM_CLASSES, m.backbone_dim).to(device)
    
    remap = {}
    for k, v in best_state.items():
        if k.startswith("distill_head."):
            continue
        if k == "dense.1.weight":
            remap["dense_conv.weight"] = v.unsqueeze(-1)
        elif k == "dense.1.bias":
            remap["dense_conv.bias"] = v
        else:
            remap[k] = v
    export_model.load_state_dict(remap, strict=False)
    
    dummy_mel = torch.randn(1, 1, 256, 313).to(device)
    onnx_path = OUT_DIR / f"sed_fold{fold_k}.onnx"
    
    torch.onnx.export(
        export_model, dummy_mel, str(onnx_path),
        input_names=["mel"],
        output_names=["clip_logits", "framewise_logits"],
        dynamic_axes={"mel": {0: "batch"}, "clip_logits": {0: "batch"}, "framewise_logits": {0: "batch"}},
        opset_version=17
    )
    print(f"  Fichier {onnx_path.name} sauvegardé avec succès.")
    
    # --- CORRECTIF : Dissociation de libération mémoire ---
    del m, export_model, perch_teacher
    gc.collect()
    torch.cuda.empty_cache()

# =================================================================
# S6 -- LANCEMENT DE LA BOUCLE SUR LES FOLDS
# =================================================================
for f in FOLDS:
    train_domain_adaptation(f)

print("\nEntraînement complet d'adaptation de domaine achevé.")
print("Vos nouveaux fichiers 'sed_fold*.onnx' sont sauvegardés dans /kaggle/working.")

Installation des bibliothèques...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 89.0 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 81.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 10.5 MB/s eta 0:00:00
Device actif : cuda
Trouvé 2 soundscapes entièrement non-étiquetés pour le pseudo-labeling.


Pseudo-labeling soundscapes:   0%|          | 0/2 [00:00<?, ?it/s]

Aucun pseudo-label généré sous le seuil de confiance spécifié.

Entraînement Fold 0...


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Perch ONNX chargé (Embed Index = 0)


NameError: name 'load_int16' is not defined